# Charlotte Projects | Week 9 · Class 4
## **Raw Drilling Telemetry: Text Parsing, Error Handling & Unit Testing**

**Interactive instructor-led notebook · 180 minutes**  
**Audience:** beginners who have completed Classes 1–3 (variables, collections, conditions, loops and functions).  
**Student deliverable:** a separate `.py` parser with tests, submitted through a Pull Request in the student's own GitHub repository.

**How to learn:** read the explanation **before** running the code, predict what will happen, execute the cell, then make the suggested change. The sample logs in this notebook are *demonstrations*, **not** the formal lab dataset. All data are synthetic. Training examples do not define safe drilling thresholds.

## Learning outcomes and teaching map

By the end, learners should be able to (1) explain why raw telemetry logs require validation, (2) split strings and turn text into typed values, (3) safely read/write text and CSV using `with`, (4) distinguish invalid input from programming bugs and use `try/except ValueError`, (5) write small `unittest` checks including expected failures, and (6) combine these into a robust script.

| Block | Topic | Time |
|---|---|---|
| A | Raw strings and structured parsing | 0–40 min |
| B | File handling and CSV output | 40–75 min |
| C | Validation and exception handling | 75–115 min |
| D | Unit testing with `unittest` | 115–150 min |
| E | Integrated walkthrough and handoff | 150–180 min |

**Prerequisite check:** `python --version` and `python -m pip install ipykernel` in the Week 9 virtual environment; open this `.ipynb` in VS Code and select that environment as the kernel. No third-party data-analysis library is needed.

---
# Part A — Raw text, fields and parsing (0–40 min)

## A1. Detailed introduction: why operational logs arrive as text

A drilling telemetry system may emit a line of characters rather than a ready-made Python dictionary. The line can represent a well ID, timestamp, pressure, units, temperature and sensor status. **Parsing** transforms this string into named pieces that a program can inspect and calculate with. **Validation** checks whether those pieces make sense for our specified training data contract. These are different steps: extracting `"3200"` is parsing; recognizing that it is a number, and that its unit is `psi`, is validation.

Why it matters: readings arrive from different instruments and software, sometimes with missing fields or inconsistent capitalization. Code that assumes every row is perfect may crash or silently create misleading reports. Our aim is a predictable process: **raw string → fields → validated values → normalized record**.

For teaching, each record has **7 pipe-separated fields**, in this order:
`well_id|timestamp|pressure|pressure_unit|temperature|temperature_unit|status`.

Pipes (`|`) are delimiters—not part of any measurement. The data contract is a **classroom convention**, not an industry-mandated log specification. We keep field positions fixed so students can reason about the data.

In [ ]:
raw = "WELL-DEMO|2026-11-10T12:00:00|3000|psi|176|F|ACTIVE"
print(raw)
print("Python type:", type(raw).__name__)
print("Number of pipe symbols:", raw.count("|"))

## A2. Detailed introduction: strings, indexing, `split()` and `strip()`

A Python `str` stores text, even when its characters look like a number. The method `split("|")` returns a **list of substrings**. We can unpack that list into descriptive variables if it contains exactly seven elements. `strip()` removes spaces or line-ending characters from the beginning and end of a field, preventing a reading such as `" psi "` from failing a comparison with `"psi"`.

**Important:** `split()` does not convert text into numeric types. `"3000"` is still a string until `float("3000")` runs. Never assume an input has exactly seven fields without checking: unpacking the wrong number raises a `ValueError`.

In [ ]:
raw = " WELL-DEMO | 2026-11-10T12:00:00 | 3000 | psi | 176 | F | ACTIVE "
parts = [part.strip() for part in raw.split("|")]
print(parts)
print("Field count:", len(parts))
print("Pressure text:", repr(parts[2]), "type:", type(parts[2]).__name__)
pressure = float(parts[2])
print("Numeric pressure:", pressure, "type:", type(pressure).__name__)

### Common mistakes and a quick prediction

- Splitting on commas when the log uses pipes returns a single large field.
- `parts[2] + 100` fails because `parts[2]` is text; use `float(parts[2])` first.
- `strip()` removes surrounding whitespace, not internal spaces.
- Strings are **case-sensitive**: `"active" != "ACTIVE"`; normalize with `.upper()` when the contract allows it.

**Try A1:** Change the line to contain spaces around `psi`. Why does `strip()` matter?  
**Try A2:** Delete the `status` field. How many fields do you now get? What would happen during unpacking?

In [ ]:
readings = ["3000", "2950.5", " 2800 "]
print([float(text.strip()) for text in readings])
print("active".upper() == "ACTIVE")

---
# Part B — Reading files and exporting CSV (40–75 min)

## B1. Detailed introduction: program memory versus persistent files

A string stored in a notebook variable disappears when the notebook kernel stops. A **file** preserves data on disk so it can be read in a later session. Python's built-in `open()` can read a text file line by line. Use `with open(..., encoding="utf-8") as handle:`: the context manager automatically closes the file when the block finishes, including when an error occurs.

The `pathlib.Path` class represents file paths without hard-coding Windows or macOS separators. Paths can be relative to the current working directory; notebook working directories can differ, so we use a small temporary file for a self-contained demonstration.

**Why line-by-line?** Industrial logs can grow large. Iteration over a file avoids loading every line into memory at once. `enumerate(handle, start=1)` attaches line numbers so an error report can identify the specific record. Blank lines or comments (`#`) are usually ignored based on an explicitly stated processing rule.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary:
    source = Path(temporary) / "demo_logs.txt"
    source.write_text("# synthetic demo\nWELL-A|3000|psi\n\nWELL-B|2800|psi\n", encoding="utf-8")
    with source.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip() or line.lstrip().startswith("#"):
                continue
            print(f"Line {line_number}:", line.strip())

## B2. Detailed introduction: why and how to write CSV

A CSV file is a human-readable table format. Its rows are records; its columns are separated by commas. A parser can convert raw `|`-delimited text into **standardized CSV** that other tools can read. Avoid building CSV by joining fields manually: names can contain punctuation and need correct quoting. The standard-library `csv.DictWriter` takes a list of dictionaries and a declared `fieldnames` order.

Always use `newline=""` when opening CSV files in Python and specify `encoding="utf-8"`. The file header becomes an explicit contract for the cleaned output. We will create a temporary example so running the notebook does not modify formal lab files.

In [ ]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

clean_records = [
    {"well_id": "WELL-A", "pressure_kpa": 20684.27, "status": "ACTIVE"},
    {"well_id": "WELL-B", "pressure_kpa": 19305.32, "status": "OFFLINE"},
]
with TemporaryDirectory() as temporary:
    output = Path(temporary) / "demo_clean.csv"
    with output.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["well_id", "pressure_kpa", "status"])
        writer.writeheader()
        writer.writerows(clean_records)
    print(output.read_text(encoding="utf-8"))

**Try B1:** Add a third dictionary to `clean_records` and rerun.  
**Discussion:** Why should a production parser write rejected rows to a separate audit file instead of silently discarding them? In this class, we return reasons for rejection; a later system could persist that audit data securely.

---
# Part C — Data validation and exceptions (75–115 min)

## C1. Detailed introduction: malformed data is not the same as a software bug

Realistic telemetry may contain bad characters (`"three thousand"`), blank readings, unsupported units, extra fields, or impossible values under the chosen classroom rules. Calling `float("bad")` raises a built-in **`ValueError`**. A robust parser should reject the bad row, explain the reason, and continue with the next row when policy permits.

`try` tells Python which operation might fail. `except ValueError as exc` handles **expected invalid user/data input**. Avoid a blanket `except:` or `except Exception:` during routine parsing: those also conceal programming mistakes that instructors need to see. Use `raise ValueError("meaningful message")` to enforce your data contract deliberately.

**Engineering units:** convert psi → kPa using `6.8947572932`; Fahrenheit → Celsius using `(F - 32) * 5 / 9`. Values already in kPa or Celsius remain unchanged. A unit label like `bar` should be rejected in this exercise unless the assignment explicitly defines its conversion. Never infer a missing unit.

In [ ]:
examples = ["3000", "bad", " 2750 ", ""]
for text in examples:
    try:
        number = float(text)
    except ValueError:
        print(repr(text), "-> rejected: not numeric")
    else:
        print(repr(text), "-> valid:", number)

## C2. Detailed introduction: contract checks and a reusable parser function

Make a small function do **one coherent job**: accept one log line and return a normalized record or raise `ValueError`. A clear function is easier to inspect, reuse and test than copying parsing statements into a large loop.

Useful steps: (1) count fields, (2) trim whitespace, (3) convert numbers, (4) verify the units, (5) apply formulas, (6) return a dictionary. In this teaching example, pressure must be non-negative. This is a *training validation rule*, not a comprehensive sensor-quality standard. A separate ingestion loop should decide what to do with a rejected line.

In [ ]:
PSI_TO_KPA = 6.8947572932

def normalize_pressure(value_text: str, unit: str) -> float:
    value = float(value_text)  # ValueError for invalid numbers
    if value < 0:
        raise ValueError("negative pressure not allowed in this exercise")
    if unit.strip().lower() == "psi":
        return round(value * PSI_TO_KPA, 2)
    if unit.strip().lower() == "kpa":
        return round(value, 2)
    raise ValueError(f"unsupported pressure unit: {unit}")

for value_text, unit in [("3000", "psi"), ("20000", "kPa"), ("3100", "bar")]:
    try:
        print(value_text, unit, "=>", normalize_pressure(value_text, unit), "kPa")
    except ValueError as exc:
        print("Rejected:", exc)

## C3. Detailed introduction: communicate rejection reasons precisely

A useful error message states **what failed** without leaking sensitive information. For example, `"expected 7 fields; received 6"` is better than `"bad row"`. The ingestion loop can pair that error with the original line number. This creates an audit trail instructors can inspect.

Never solve invalid data by replacing it with zero unless your data contract explicitly requires imputation: a fabricated zero can skew calculations. Do not treat offline readings as failed parsing automatically; status can be a valid field that downstream analysis handles separately. In the formal lab, both `ACTIVE` and `OFFLINE` are valid statuses.

In [ ]:
demo_rows = ["WELL-A|3000|psi", "WELL-B|bad|psi", "WELL-C|2500|bar", "WELL-D|2800"]
accepted = []
rejected = []
for number, raw in enumerate(demo_rows, start=1):
    try:
        fields = [piece.strip() for piece in raw.split("|")]
        if len(fields) != 3:
            raise ValueError(f"expected 3 demo fields; received {len(fields)}")
        well, value, unit = fields
        accepted.append({"well_id": well, "pressure_kpa": normalize_pressure(value, unit)})
    except ValueError as exc:
        rejected.append((number, str(exc)))
print("ACCEPTED:", accepted)
print("REJECTED:", rejected)

**Try C1:** Replace `"bad"` with `"2950"`. Predict the accepted/rejected counts.  
**Try C2:** Enter the unit `" PSI "`. What happens and why?  
**Instructor checkpoint:** Ask students to explain why `except ValueError` belongs around the individual row processing rather than the entire batch.

---
# Part D — Unit testing with `unittest` (115–150 min)

## D1. Detailed introduction: what is a unit test?

A **unit test** is a short, repeatable check of a small behavior: given a known input, does a function produce an expected output or an expected error? Unit tests catch regressions when code is changed. They are **not** proof that industrial code is safe, and they do not replace domain validation.

Python ships with `unittest`, so nothing new needs to be installed. Test cases are methods named `test_...` in a class that inherits from `unittest.TestCase`. Assertions state expectations: `assertEqual` compares exact results, `assertAlmostEqual` suits floating-point values, and `assertRaises` verifies an expected exception.

Good tests cover: a normal conversion, a unit already in its destination system, a malformed numeric input, an unsupported unit, and an important boundary such as negative pressure. Tests should be independent and deterministic, with no network calls or user inputs.

In [ ]:
import unittest

class PressureConversionTests(unittest.TestCase):
    def test_psi_conversion(self):
        self.assertAlmostEqual(normalize_pressure("100", "psi"), 689.48, places=2)

    def test_metric_passthrough(self):
        self.assertEqual(normalize_pressure("20000", "kPa"), 20000)

    def test_invalid_number(self):
        with self.assertRaises(ValueError):
            normalize_pressure("n/a", "psi")

    def test_unsupported_unit(self):
        with self.assertRaises(ValueError):
            normalize_pressure("100", "bar")

    def test_negative_value(self):
        with self.assertRaises(ValueError):
            normalize_pressure("-5", "psi")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(PressureConversionTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), "Demo tests should pass"

## D2. Detailed introduction: testing a script outside the notebook

The notebook demonstrates ideas, but **formal assessment must be a Python script**. Students will create `test_student_telemetry_parser.py` alongside `student_telemetry_parser.py`, and run tests from the VS Code terminal:

```bash
python -m unittest discover -v
```

This discovers files matching `test*.py` and executes their `test_...` methods. A failing test is feedback: inspect the failure message, fix the code, and rerun. Do not simply delete a test to make the suite green. For the Class 4 submission, tests must include both successful conversions and malformed inputs.

**Try D1:** Add a sixth test that checks `normalize_pressure("0", "psi") == 0`.  
**Try D2:** Temporarily change the conversion constant and rerun the tests. Which assertion detects the regression? Restore the constant afterwards.

---
# Part E — Integrated demonstration and independent task (150–180 min)

## E1. Detailed introduction: separate responsibilities in a reliable program

A robust parser is easiest to reason about in three layers: **(1) parse one record**, **(2) process the file**, and **(3) test behaviors**. `parse_line()` should not open files; `process_log()` should not repeat conversion formulas; tests should call functions with known inputs and expected results.

The integration demonstration below processes a *small, simplified, three-field example*, deliberately different from the seven-field student lab. Students will adapt the design—not copy a completed solution—to meet the full contract in their `.py` starter script.

Flow: `open source → skip comments → parse/validate per line → collect accepted rows → report rejected row numbers → write CSV → run tests`. Changes to any one part should be testable without rewriting the entire application.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv

def parse_demo_record(raw: str) -> dict:
    fields = [item.strip() for item in raw.split("|")]
    if len(fields) != 3:
        raise ValueError("demo requires exactly 3 fields")
    well, pressure, unit = fields
    if not well:
        raise ValueError("missing well ID")
    return {"well_id": well, "pressure_kpa": normalize_pressure(pressure, unit)}

with TemporaryDirectory() as temp:
    source = Path(temp) / "input.txt"
    destination = Path(temp) / "clean.csv"
    source.write_text("# training only\nWELL-DEMO-1|3100|psi\nWELL-DEMO-2|oops|psi\nWELL-DEMO-3|20500|kPa\n", encoding="utf-8")
    ok, bad = [], []
    for number, line in enumerate(source.read_text(encoding="utf-8").splitlines(), start=1):
        if not line or line.startswith("#"):
            continue
        try:
            ok.append(parse_demo_record(line))
        except ValueError as exc:
            bad.append((number, str(exc)))
    with destination.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["well_id", "pressure_kpa"])
        writer.writeheader()
        writer.writerows(ok)
    print("Valid rows:", len(ok), "Rejected:", bad)
    print(destination.read_text(encoding="utf-8"))

## E2. Formal student lab and GitHub handoff

The assessment files are in **`Student_Practical_Files/`**. They contain a *different* synthetic dataset with seven required fields and six intentional malformed rows. Your independent task is to:

1. Implement `parse_line()` and `process_log()` in `student_telemetry_parser.py` without using the instructor solution.
2. Convert pressure to **kPa** and temperature to **°C**; preserve well ID, ISO timestamp and status.
3. Reject and report malformed rows with line numbers. Continue processing valid rows.
4. Produce `validated_telemetry.csv` locally and write **at least five `unittest` tests** in `test_student_telemetry_parser.py`.
5. In your **own** Week 9 GitHub repository, work on `feature/class-04`, commit code and tests, open a Pull Request to `main`, and submit the link.

**Do not commit:** `.venv/`, `.env`, `__pycache__/`, real operational data, or instructor solutions. The notebook may be kept as personal notes; the assessed work is the `.py` files and PR.

### Exit ticket
- Explain the difference between parsing and validation.
- Identify one `ValueError` that a log parser should catch and one programming bug it should *not* hide.
- Explain what `assertRaises` proves.
- Explain why CSV output needs a fixed header order.

**Next class:** integrate these skills into a larger Week 9 telemetry script capstone.